# Input-calibration plotter

Compares FC telemetry (MAVSDK Position/Quaternion/Velocity/IMU) and Gazebo
ground-truth poses against the commanded attitude-rate + thrust profile sent
by `input_calibration.py`.

Modeled after `~/ws/scripts/soft_precise_landing/plotter_input_calibration.ipynb`
but reads from `PX4_Gazebo/calibration_data/input/<timestamp>/`.

Frame conventions (all data normalized to PX4 NED + body FRD throughout):
- **W** = world, NED (x=North, y=East, z=Down) — PX4 native; Gazebo ENU is converted via `NED_from_ENU`
- **B / I** = body, FRD (Forward/Right/Down) — PX4 native and MAVSDK PositionBody; Gazebo FLU is converted via `FRD_2_FLU = DCM(x=180°)`

### Manuscript IBVS terminology (only `w` shows up in this notebook)

The PLASMC manuscript uses `s`, `ṡ`, `h`, `w` for the IBVS image-plane quantities (see the output-calibration plotter for the full glossary). This input-calibration plotter deals with physical UAV body quantities — position, velocity, acceleration, angular velocity, thrust — most of which have no direct image-plane analogue.

The one quantity that **does** map: **body angular velocity in body-FRD = virtual image angular velocity `w`** (because `R_V_from_body = I` after the SDF + cv2 chain). So:

| Variable | Manuscript meaning |
|---|---|
| `B_w_ug` (GT-derived UAV body ω, FRD) | `w_gt` (virtual image angular velocity, ground truth) |
| `w_t` (IMU body ω, FRD) | `w_meas` (virtual image angular velocity, measured by PX4 IMU) |
| `w_u` (commanded body rate, FRD) | `w_cmd` (commanded virtual image angular velocity) |
| `B_w_ut` (TEL body ω, FRD via MAVSDK) | `w_tel` (= `w_meas` post-EKF) |

Position, linear velocity, and acceleration plots are body-frame physical quantities — they don't reuse `s` or `h` (those are image-plane).

In [ ]:
import os, ast
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter as sgf
from ahrs import Quaternion, DCM

np.set_printoptions(precision=2, suppress=True)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Pick the most recent input-calibration recording, or set RUN_DIR explicitly.
PARENT = '/home/shubham/Soft-Precise-Landing/PX4_Gazebo/calibration_data/input'
_cands = [d for d in os.listdir(PARENT) if os.path.isdir(os.path.join(PARENT, d))]
RUN_DIR = os.path.join(PARENT, max(_cands, key=lambda d: os.path.getmtime(os.path.join(PARENT, d))))
print(f'Loading from: {RUN_DIR}')

tel  = np.load(f'{RUN_DIR}/Telemetry_Data.npy', allow_pickle=True)[()]
gt   = np.load(f'{RUN_DIR}/Ground_Truth.npy',   allow_pickle=True)[()]
try:
    img = np.load(f'{RUN_DIR}/Img_Data.npy', allow_pickle=True)[()]
except FileNotFoundError:
    img = None

print('tel keys:', list(tel.keys()))
print('gt  keys:', list(gt.keys()))
print(f'gt samples: {len(gt["Time"])}, duration {gt["Time"][-1] - gt["Time"][0]:.2f}s')

## Frame conventions

In [ ]:
FRD_2_FLU    = np.array(DCM(x=180.0))                           # FRD ↔ FLU (self-inverse)
FLU_2_FRD    = FRD_2_FLU                                        # alias for the same DCM
NED_from_ENU = np.array([[0.0, 1.0, 0.0],
                          [1.0, 0.0, 0.0],
                          [0.0, 0.0, -1.0]])                     # NED ↔ ENU (self-inverse)
mass = 2.114   # kg, Holybro X500 (matches MATLAB Constants.m)
g_scalar = 9.8

## Telemetry data (MAVSDK)

PositionBody / Quaternion / VelocityBody / AngularVelocityBody at odometry rate (~60 Hz).

In [ ]:
# Telemetry-side: position, velocity, Euler, angular velocity → body-FRD.
# PX4 telemetry is already in NED world + FRD body, so just project NED
# positions/velocities through inv(R_FRD_NED) into body-FRD. Euler from
# q_PX4 is FRD-based (no FRD_2_FLU correction).
start_idx_o = np.searchsorted(tel['Odometry Timestamp'], gt['Start Time'])
positions     = tel['Position Body'][start_idx_o:]
quaternions   = tel['Quaternion'][start_idx_o:]
velocities    = tel['Velocity Body'][start_idx_o:]
ang_vels      = tel['Angular Velocity Body'][start_idx_o:]
t_o = np.array(tel['Odometry Timestamp'][start_idx_o:]) - gt['Start Time']

# Body-frame delta-rotation + Tait-Bryan extraction utilities used by both
# TEL and GT Euler paths.
def _R_delta_body(R_spawn, R_now):
    """Body-frame delta rotation: takes vectors in spawn-body to current-body."""
    return R_spawn.T @ R_now

def _dcm_to_euler_zyx(R):
    """Tait-Bryan ZYX intrinsic (yaw-pitch-roll) from a body→world DCM."""
    return np.array([
        np.arctan2(R[2, 1], R[2, 2]),   # roll
        np.arcsin(np.clip(-R[2, 0], -1.0, 1.0)),   # pitch
        np.arctan2(R[1, 0], R[0, 0]),   # yaw
    ])

# TEL spawn: first PX4 quaternion → R_FRD_NED (already in target convention).
q0 = quaternions[0]
R_t_spawn = Quaternion([q0.w, q0.x, q0.y, q0.z]).to_DCM()

n_o = len(positions)
B_x_ut = np.zeros((n_o, 3))   # UAV position (body-FRD)
EA_t   = np.zeros((n_o, 3))   # Euler angles (FRD/NED) delta from spawn
I_R_B  = np.zeros((n_o, 3, 3))    # body-FRD → world-NED
for i, (pos, q) in enumerate(zip(positions, quaternions)):
    I_x_u   = np.array([pos.x_m, pos.y_m, pos.z_m])      # NED
    I_R_B[i]  = Quaternion([q.w, q.x, q.y, q.z]).to_DCM()   # R_FRD_NED
    R_delta   = _R_delta_body(R_t_spawn, I_R_B[i])
    EA_t[i]   = _dcm_to_euler_zyx(R_delta)
    B_x_ut[i] = np.linalg.inv(I_R_B[i]) @ I_x_u           # NED → body-FRD
# Unwrap yaw to avoid ±π discontinuities (sustained input-cal rate commands
# can accumulate >360°).
EA_t[:, 2] = np.unwrap(EA_t[:, 2])

B_v_ut = np.zeros((n_o, 3))
B_w_ut = np.zeros((n_o, 3))
for i, (R, vel, w) in enumerate(zip(I_R_B, velocities, ang_vels)):
    B_v_ut[i] = np.linalg.inv(R) @ np.array([vel.x_m_s, vel.y_m_s, vel.z_m_s])   # NED → body-FRD
    B_w_ut[i] = np.array([w.roll_rad_s, w.pitch_rad_s, w.yaw_rad_s])              # body-FRD (PX4 native)

print(f'telemetry samples: {n_o}, t in [{t_o[0]:.2f}, {t_o[-1]:.2f}]s')

## IMU data

Acceleration (FRD: forward/right/down) and angular velocity (FRD) — these come directly from PX4's IMU; useful for cross-checking and for computing thrust from `T = m*(a_z + g)`.

In [ ]:
start_idx_a = np.searchsorted(tel['IMU Timestamp'], gt['Start Time'])
t_a = np.array(tel['IMU Timestamp'][start_idx_a:]) - gt['Start Time']
a_t = np.array([[a.forward_m_s2, a.right_m_s2, a.down_m_s2]
                for a in tel['Acceleration'][start_idx_a:]])
w_t = np.array([[w.forward_rad_s, w.right_rad_s, w.down_rad_s]
                for w in tel['Angular Velocity FRD'][start_idx_a:]])
# Thrust from IMU: T = m*(a_down + g)
T_t = mass * (a_t[:, 2] + g_scalar)
print(f'IMU samples: {len(t_a)}')

## Ground truth (Gazebo `/pose`)

Compute UAV pose in body frame, then differentiate world-frame position twice for velocity and acceleration. Heavy savgol smoothing on the derivatives (typical sgf(301, 2) for v, sgf(301, 2) for a — matches the reference notebook).

In [ ]:
# Ground-truth side: Gazebo ENU/FLU → PX4 NED/FRD at the source so all
# world-frame intermediates (W_x_u, W_v_u, W_a_u) are NED and body-FRD
# outputs (B_x_ug, B_v_ug, B_a_ug) match the telemetry side.
#   R_FRD_NED = NED_from_ENU @ R_FLU_ENU @ FRD_2_FLU
#   N_x_NED   = NED_from_ENU @ ENU_pos
# This is mathematically equivalent to the old "leave intermediates in
# ENU, compose inv(R_FRD_ENU) at projection time" pattern — but the
# intermediates now carry the same frame the telemetry side reports.
t_g = np.array(gt['Time'])
uav_poses = gt['UAV Pose']
n_g = len(uav_poses)

# GT spawn for Euler: use the first UAV pose (synchronized with TEL spawn).
# gt['Start Pose'] is the ground spawn (pre-takeoff) and won't match the TEL
# reference, which starts post-takeoff after the search-sorted index.
q0_g = uav_poses[0].orientation
R_g_spawn_FLU_ENU = Quaternion([q0_g.w, q0_g.x, q0_g.y, q0_g.z]).to_DCM()
R_g_spawn_NED_FRD = NED_from_ENU @ R_g_spawn_FLU_ENU @ FRD_2_FLU    # body-FRD → world-NED at spawn

W_R_B = np.zeros((n_g, 3, 3))     # body-FRD → world-NED, per sample
W_x_u = np.zeros((n_g, 3))        # UAV world position, NED
EA_g  = np.zeros((n_g, 3))
for i, p in enumerate(uav_poses):
    q = Quaternion([p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z])
    R_now_FLU_ENU = q.to_DCM()
    W_R_B[i] = NED_from_ENU @ R_now_FLU_ENU @ FRD_2_FLU            # R_FRD_NED
    W_x_u[i] = NED_from_ENU @ np.array([p.position.x, p.position.y, p.position.z])  # NED
    # Body-frame delta-from-spawn rotation, Euler extracted ZYX (matches TEL).
    R_delta  = _R_delta_body(R_g_spawn_NED_FRD, W_R_B[i])
    EA_g[i]  = _dcm_to_euler_zyx(R_delta)
EA_g[:, 2] = np.unwrap(EA_g[:, 2])

# Interpolate GT to a uniform timebase before differentiating so bridge-
# cadence jitter (std~3ms, max-gap ~92ms) doesn't leak into the derived
# velocity / acceleration. Then sgf is applied on the regular grid
# (where it's actually correct — sgf assumes uniform dt).
from scipy.interpolate import interp1d as _interp1d
if n_g >= 5:
    t_g_unif    = np.linspace(t_g[0], t_g[-1], n_g)
    W_x_u_unif  = np.column_stack([
        _interp1d(t_g, W_x_u[:, k], fill_value='extrapolate', bounds_error=False)(t_g_unif)
        for k in range(3)
    ])
    # Adaptive sgf window: target ~2 Hz cutoff (sgf(W,p) ~ Butterworth-like
    # with cutoff ≈ 0.45*fs/W). Previously fixed W=5 → 11 Hz cutoff at the
    # new 125 Hz GT rate, barely filtered anything.
    _dt_med = float(np.median(np.diff(t_g_unif)))
    _W = max(5, int(round(0.225 / _dt_med)) | 1)
    if _W % 2 == 0: _W += 1
    W_x_u_filt  = sgf(W_x_u_unif, _W, 3, axis=0)
    W_v_u_unif  = np.gradient(W_x_u_filt, t_g_unif, axis=0)
    W_v_u_filt2 = sgf(W_v_u_unif, _W, 3, axis=0)
    W_a_u_unif  = np.gradient(W_v_u_filt2, t_g_unif, axis=0)
    # Back to original t_g for downstream array alignment.
    W_v_u = np.column_stack([
        _interp1d(t_g_unif, W_v_u_unif[:, k], fill_value='extrapolate', bounds_error=False)(t_g)
        for k in range(3)
    ])
    W_a_u = np.column_stack([
        _interp1d(t_g_unif, W_a_u_unif[:, k], fill_value='extrapolate', bounds_error=False)(t_g)
        for k in range(3)
    ])
else:
    W_v_u = np.gradient(W_x_u, t_g, axis=0)
    W_a_u = np.gradient(W_v_u, t_g, axis=0)

B_x_ug = np.einsum('ijk,ik->ij', np.linalg.inv(W_R_B), W_x_u)   # NED → body-FRD
B_v_ug = np.einsum('ijk,ik->ij', np.linalg.inv(W_R_B), W_v_u)   # NED → body-FRD
B_a_ug = np.einsum('ijk,ik->ij', np.linalg.inv(W_R_B), W_a_u)   # NED → body-FRD

In [ ]:
# BODY_YAW_SOURCE: pick which rotation matrix is used to project TEL's
# world position/velocity into body frame.
#
# 'gt' (default) — project TEL world position through GT's rotation. This
#   isolates TEL EKF world-position error from EKF yaw error; under aggressive
#   input-cal maneuvers, TEL EKF yaw drifts 30-46°, which (when applied to a
#   130m world position) inflates the body-frame error from ~1m to ~100m.
#   The 'gt' projection gives clean TEL/GT body-frame overlay (~1m residual,
#   which is the actual world-position EKF error).
#
# 'ekf' — project TEL world position through TEL's own EKF rotation (legacy).
#   Useful as a diagnostic: any body-frame TEL/GT divergence here is the
#   combined effect of position and yaw EKF errors. Reach for this when you
#   want to see EKF yaw drift in the position plot.
import os
BODY_YAW_SOURCE = os.environ.get('BODY_YAW_SOURCE', 'gt')
if BODY_YAW_SOURCE == 'gt':
    from scipy.interpolate import interp1d as _interp1d
    # Build per-TEL-sample GT quaternion via component interp + unit-norm,
    # then convert it to R_FRD_NED in one step (same convention as cell 10's
    # W_R_B). Both TEL and GT world frames are NED here, so no ENU↔NED swap
    # is needed at projection time.
    q_g_arr = np.array([[u.orientation.w, u.orientation.x,
                         u.orientation.y, u.orientation.z] for u in uav_poses])
    q_g_at_o = np.column_stack([_interp1d(t_g, q_g_arr[:, k],
                                          fill_value='extrapolate',
                                          bounds_error=False)(t_o)
                                for k in range(4)])
    q_g_at_o /= np.linalg.norm(q_g_at_o, axis=1, keepdims=True)
    R_g_at_o_FRD_NED = np.array([
        NED_from_ENU @ Quaternion(q).to_DCM() @ FRD_2_FLU for q in q_g_at_o
    ])
    NED_pos  = np.array([[p.x_m, p.y_m, p.z_m] for p in positions])
    v_w_NED  = np.array([[v.x_m_s, v.y_m_s, v.z_m_s] for v in velocities])
    B_x_ut   = np.einsum('ijk,ik->ij', np.linalg.inv(R_g_at_o_FRD_NED), NED_pos)
    B_v_ut   = np.einsum('ijk,ik->ij', np.linalg.inv(R_g_at_o_FRD_NED), v_w_NED)
    print(f"[BODY_YAW_SOURCE=gt] TEL body-frame re-projected through GT R (NED)")
    print(f"  Eliminates TEL EKF yaw-error contribution; residual is world-pos error only.")
else:
    print(f"[BODY_YAW_SOURCE=ekf] TEL body-frame uses TEL EKF R (legacy)")

In [ ]:
# Body angular velocity via QUATERNION DIFFERENCE (preserves SO(3)).
# Previously: filtered the 9 elements of W_R_B then np.gradient. That
# differentiates each matrix element independently and the result is NOT a
# skew-symmetric matrix — the non-skew leakage contaminates the extracted
# angular velocity (over-reports ω_z by ~2× on Gazebo recordings; see
# plotter_output_calibration.ipynb cell 6 diagnosis).
#
# Correct formula uses the unit-quaternion structure directly:
#   δq = conj(q[i-1]) * q[i+1]   (relative rotation, body frame at t[i-1])
#   ω_body ≈ 2 · δq.xyz / (t[i+1] - t[i-1])
# The quaternion in uav_poses[*].orientation is "FLU body → ENU world"; the
# extracted ω is in FLU body frame, then converted to FRD with FLU_2_FRD.
FLU_2_FRD = FRD_2_FLU                       # DCM(x=180) is self-inverse

def _body_omega_from_quats(quats, t):
    N = len(quats); w = np.zeros((N, 3))
    for i in range(N):
        i0 = max(0, i - 1); i1 = min(N - 1, i + 1)
        if i1 == i0: continue
        dt_pair = t[i1] - t[i0]
        if dt_pair < 1e-9: continue
        w0, x0, y0, z0 = quats[i0]
        w1, x1, y1, z1 = quats[i1]
        dq_x = w0*x1 - x0*w1 - y0*z1 + z0*y1
        dq_y = w0*y1 + x0*z1 - y0*w1 - z0*x1
        dq_z = w0*z1 - x0*y1 + y0*x1 - z0*w1
        w[i] = 2.0 * np.array([dq_x, dq_y, dq_z]) / dt_pair
    return w

uav_quats = np.array([[p.orientation.w, p.orientation.x,
                       p.orientation.y, p.orientation.z] for p in uav_poses])
B_w_ug = (FLU_2_FRD @ _body_omega_from_quats(uav_quats, t_g).T).T
nans, idx = np.isnan(B_w_ug), lambda z: z.nonzero()[0]
B_w_ug[nans] = np.interp(idx(nans), idx(~nans), B_w_ug[~nans])

## Input commands

`gt['Command']` has shape `(N, 4)` — rows are `[roll_rate, pitch_rate, yaw_rate, thrust_delta_N]`.
The thrust delta enters `convert_2_sys_cmd` as `throttle = 0.738 - thrust/45`.

In [ ]:
cmds = np.array(gt['Command'])
w_u = cmds[:, :3]   # commanded rate (rad/s)
T_u = cmds[:, 3]    # commanded thrust delta (N)
print(f'unique commands:\n{np.unique(np.round(cmds, 3), axis=0)}')

## Plots

Each subplot: telemetry vs ground truth (and command where applicable).

### Position (body frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['x', 'y', 'z'])):
    ax.plot(t_o, B_x_ut[:, i], label='Telemetry')
    ax.plot(t_g, B_x_ug[:, i], label='Ground Truth')
    ax.set(title=f'Position {name} (body, m)', xlabel='t (s)', ylabel=f'{name} (m)')
    ax.legend()
plt.show()

### Euler angles

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['roll', 'pitch', 'yaw'])):
    ax.plot(t_o, EA_t[:, i], label='Telemetry')
    ax.plot(t_g, EA_g[:, i], label='Ground Truth')
    ax.set(title=f'{name} (rad)', xlabel='t (s)', ylabel=f'{name} (rad)')
    ax.legend()
plt.show()

### PX4 EKF yaw vs GT yaw — drift during aggressive maneuvers

Both traces show yaw delta-from-spawn (the quaternion-correct form). Any divergence between them is PX4's EKF yaw-estimation error — it integrates gyro short-term but corrects slowly via magnetometer, so under sustained input-cal rate commands the EKF yaw drifts away from truth. This drift, multiplied by the drone's world-position magnitude, is what produces the body-frame position mismatch (see `BODY_YAW_SOURCE` cell).

In [ ]:
from scipy.interpolate import interp1d as _interp1d
# Interpolate GT yaw onto TEL timebase for per-sample error
_yg_at_o = _interp1d(t_g, EA_g[:, 2],
                     fill_value='extrapolate', bounds_error=False)(t_o)
_err_deg = np.degrees(EA_t[:, 2] - _yg_at_o)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True,
                          gridspec_kw={'height_ratios': [2, 1]})
axes[0].plot(t_o, np.degrees(EA_t[:, 2]),
              label='TEL EKF yaw delta (magnetometer-derived)',
              linewidth=1.5, color='tab:blue')
axes[0].plot(t_g, np.degrees(EA_g[:, 2]),
              label='GT yaw delta (Gazebo truth)',
              linewidth=2.0, color='tab:orange')
axes[0].set_title('PX4 EKF yaw vs GT yaw — drift during aggressive input-cal maneuvers')
axes[0].set_ylabel('Yaw (deg, delta from spawn)')
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(t_o, _err_deg, color='tab:red', linewidth=1.2)
axes[1].axhline(0, linestyle='--', color='grey', alpha=0.6)
axes[1].fill_between(t_o, 0, _err_deg, alpha=0.2, color='tab:red')
axes[1].set_title(
    f'TEL − GT yaw error  '
    f'(mean = {_err_deg.mean():+.1f}°, max |Δ| = {np.max(np.abs(_err_deg)):.1f}°)'
)
axes[1].set_xlabel('t (s)'); axes[1].set_ylabel('Δ yaw (deg)')
axes[1].grid(True, alpha=0.3)
plt.show()


### Linear velocity (body frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['vx', 'vy', 'vz'])):
    ax.plot(t_o, B_v_ut[:, i], label='Telemetry')
    ax.plot(t_g, B_v_ug[:, i], label='Ground Truth')
    ax.set(title=f'Velocity {name} (body, m/s)', xlabel='t (s)', ylabel=f'{name} (m/s)')
    ax.legend()
plt.show()

### Virtual image angular velocity `w` — IMU vs command

The most direct input-calibration plot: how does PX4's measured body rate (IMU, `w_meas` ≡ manuscript `w`) compare to the commanded rate (`w_cmd`)? Since `R_V_from_body = I`, this body-FRD ω is identical to the virtual image angular velocity that downstream PLASMC uses in `ṡ = L · [h; w]`.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['x', 'y', 'z'])):
    ax.plot(t_a, w_t[:, i], label=r'$w_{\mathrm{meas}}$ (PX4 IMU)')
    ax.plot(t_g, w_u[:, i], label=r'$w_{\mathrm{cmd}}$ (commanded rate)',
             linestyle='--', linewidth=2)
    ax.set(title=f'Virtual image angular velocity $w_{name}$ (body-FRD, rad/s)',
           xlabel='t (s)', ylabel=f'$w_{name}$ (rad/s)')
    ax.legend()
plt.show()

### Linear acceleration

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
g_vec = [0, 0, g_scalar]
for i, (ax, name) in enumerate(zip(axes, ['ax', 'ay', 'az'])):
    # IMU acceleration in body FRD, with gravity removed
    ax.plot(t_a, -a_t[:, i] - g_vec[i], label='Telemetry IMU (g-removed)')
    ax.plot(t_g, B_a_ug[:, i], label='Ground Truth (d²x/dt²)')
    if i == 2:
        # Thrust component on z: T/m maps to -a_z in body FRD
        ax.plot(t_g, -T_u / mass, label='Commanded thrust / m', linestyle='--')
    ax.set(title=f'Acceleration {name} (body, m/s²)',
           xlabel='t (s)', ylabel=f'{name} (m/s²)')
    ax.legend()
plt.show()

### Body thrust

`T_t = mass * (a_down + g)` from IMU vs `T_u` commanded (delta over hover; total throttle in `convert_2_sys_cmd` is `0.738 − T_u/45`).

In [ ]:
plt.figure(figsize=(12, 5), constrained_layout=True)
plt.plot(t_a, T_t, label='Telemetry (m·(a_z + g))')
plt.plot(t_g, T_u, label='Commanded thrust delta (N)', linestyle='--')
plt.title('Body thrust (N)')
plt.xlabel('t (s)'); plt.ylabel('T (N)')
plt.legend(); plt.show()

## Command → response quality

Per-axis Pearson correlation between commanded virtual image angular velocity `w_cmd` and IMU-measured `w_meas` (= manuscript `w`), computed only where the command is non-zero (so idle hover periods don't dominate the metric). Plus thrust command vs IMU-derived thrust.

In [ ]:
# Resample command onto IMU timeline
w_u_on_imu = np.array([np.interp(t_a, t_g, w_u[:, i]) for i in range(3)]).T
T_u_on_imu = np.interp(t_a, t_g, T_u)

def corr_active(meas, cmd, threshold):
    m = np.abs(cmd) > threshold
    if m.sum() < 10: return float('nan')
    c = np.corrcoef(meas[m], cmd[m])[0, 1]
    return c

# w_meas (IMU) vs w_cmd correlation per axis — manuscript w = virtual image angular velocity.
print('Virtual image angular velocity w_cmd vs w_meas (IMU) — only where |w_cmd| > 0.01 rad/s:')
for i, name in enumerate(['w_x', 'w_y', 'w_z']):
    c = corr_active(w_t[:, i], w_u_on_imu[:, i], 0.01)
    rmse = np.sqrt(np.nanmean((w_t[:, i] - w_u_on_imu[:, i])**2))
    print(f'  {name}: corr={c:+.3f}  RMSE={rmse:.4f} rad/s')

print('\nThrust command vs IMU-derived (only when |cmd| > 0.1 N):')
c = corr_active(T_t, T_u_on_imu, 0.1)
rmse = np.sqrt(np.nanmean((T_t - T_u_on_imu)**2))
print(f'  T: corr={c:+.3f}  RMSE={rmse:.4f} N   (hover T_t ≈ {mass*g_scalar:.2f} N)')